# Oil News Project — Demo Notebook

End-to-end walkthrough of the Brent crude oil price prediction pipeline:

1. **Setup** — working directory, import paths, Plotly renderer
2. **Database Configuration** — MySQL connection via `.env`
3. **Load Datasets** — CSV → MySQL upsert + analytics views
4. **Train Model** — `StandardScaler → Ridge` with a 1-day horizon
5. **Forward Forecast** — iterative 10-day Brent price projection
6. **Visualizations** — interactive Plotly charts (zoomable, pannable)

> **Run cells top-to-bottom.**  MySQL must be reachable for step 3;
> steps 4–6 only need the CSV datasets in `datasets/`.

## 1. Setup
Normalises the working directory, adds `src/` to the Python import path, and
configures the Plotly renderer so `fig.show()` produces interactive HTML charts
rather than static images.

In [ ]:
import sys, os
from pathlib import Path

# ── Working directory ─────────────────────────────────────────────────────────
# All scripts assume they are run from the project root.  Adjust CWD if the
# notebook kernel started somewhere else (e.g. inside Visualization/).
_root = Path.cwd()
if not (_root / "src").exists() and (_root.parent / "src").exists():
    os.chdir(_root.parent)
    _root = Path.cwd()

# ── Import path ───────────────────────────────────────────────────────────────
# Add src/ so db_config and friends can be imported without installation.
_src = str(_root / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

# ── Plotly renderer ───────────────────────────────────────────────────────────
# Force the interactive HTML renderer.  Without this, Jupyter may auto-detect
# the wrong backend and fall back to a static PNG image instead of a live chart.
import plotly.io as pio
pio.renderers.default = "notebook"

print(f"Project root : {_root}")
print(f"Python       : {sys.version.split()[0]}")
print(f"Plotly render: {pio.renderers.default}")

## 2. Database Configuration
Loads MySQL connection details from `.env` (or environment variables).

Source: `src/db_config.py`

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class MySQLConfig:
    host: str
    port: int
    user: str
    password: str
    database: str


def load_dotenv(path: Path = Path(".env")) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def get_mysql_config() -> MySQLConfig:
    load_dotenv()
    return MySQLConfig(
        host=os.getenv("MYSQL_HOST", "127.0.0.1"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
        user=os.getenv("MYSQL_USER", "root"),
        password=os.getenv("MYSQL_PASSWORD", ""),
        database=os.getenv("MYSQL_DATABASE", "oil_news_project"),
    )

## 3. Load Datasets into MySQL
Reads every CSV in `datasets/` and upserts it into MySQL, then applies the
analytics-views SQL.

Source: `src/load_mysql.py`

In [ ]:
from __future__ import annotations

import argparse
import csv
import re
from collections import defaultdict
from pathlib import Path
from typing import Iterable


from db_config import get_mysql_config


DATASET_DIR = Path("datasets")

DATE_COLUMNS = {
    "trade_date",
    "event_date",
    "gpr_date",
    "month_start",
    "snapshot_date",
    "market_date",
    "full_date",
}


def require_connector():
    try:
        import mysql.connector  # type: ignore
    except ModuleNotFoundError as exc:
        raise SystemExit(
            "mysql-connector-python is required for loading MySQL. "
            "Install with: python -m pip install -r requirements.txt"
        ) from exc
    return mysql.connector


def q(identifier: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Unsafe identifier: {identifier}")
    return f"`{identifier}`"


def load_data_dictionary(path: Path) -> dict[str, dict[str, str]]:
    if not path.exists():
        return {}
    mapping: dict[str, dict[str, str]] = defaultdict(dict)
    with path.open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            mapping[row["table_name"]][row["column_name"]] = row["dtype"]
    return mapping


def mysql_type(column: str, dtype: str | None) -> str:
    if column in DATE_COLUMNS or (dtype and "datetime" in dtype):
        return "DATE"
    if dtype and "int" in dtype:
        return "INT"
    if dtype and "float" in dtype:
        return "DOUBLE"
    if column.endswith("_description") or column in {"description", "policy_response"}:
        return "TEXT"
    return "VARCHAR(512)"


def primary_key_for(table: str, columns: list[str]) -> str | None:
    candidates = [column for column in columns if column.endswith("_id") or column.endswith("_key")]
    if candidates and candidates[0] in columns:
        return candidates[0]
    if table.startswith("ops_"):
        candidate = table.removeprefix("ops_").rstrip("s") + "_id"
        return candidate if candidate in columns else None
    return None


def create_table_sql(table: str, columns: list[str], dtypes: dict[str, str]) -> str:
    pk = primary_key_for(table, columns)
    definitions = []
    for column in columns:
        col_type = mysql_type(column, dtypes.get(column))
        nullable = "NOT NULL" if column == pk else "NULL"
        definitions.append(f"  {q(column)} {col_type} {nullable}")
    if pk:
        definitions.append(f"  PRIMARY KEY ({q(pk)})")
    return f"CREATE TABLE IF NOT EXISTS {q(table)} (\n" + ",\n".join(definitions) + "\n) ENGINE=InnoDB;"


def iter_csv_rows(path: Path, columns: list[str]) -> Iterable[tuple[object, ...]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            values: list[object] = []
            for column in columns:
                value = row.get(column, "")
                values.append(None if value == "" else value)
            yield tuple(values)


def load_csv(cursor, table: str, path: Path, dtypes: dict[str, str], replace: bool) -> int:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        columns = list(next(csv.reader(handle)))

    cursor.execute(create_table_sql(table, columns, dtypes))
    if replace:
        cursor.execute(f"TRUNCATE TABLE {q(table)}")

    placeholders = ", ".join(["%s"] * len(columns))
    column_sql = ", ".join(q(column) for column in columns)
    insert_sql = f"INSERT INTO {q(table)} ({column_sql}) VALUES ({placeholders})"

    batch: list[tuple[object, ...]] = []
    total = 0
    for values in iter_csv_rows(path, columns):
        batch.append(values)
        if len(batch) >= 1000:
            cursor.executemany(insert_sql, batch)
            total += len(batch)
            batch.clear()
    if batch:
        cursor.executemany(insert_sql, batch)
        total += len(batch)
    return total


def apply_sql_file(cursor, path: Path, database: str) -> None:
    if not path.exists():
        return
    sql = path.read_text(encoding="utf-8").replace("oil_news_project", database)
    for statement in [part.strip() for part in sql.split(";") if part.strip()]:
        cursor.execute(statement)


def main() -> None:
    parser = argparse.ArgumentParser(description="Load workspace CSV datasets into MySQL.")
    parser.add_argument("--dataset-dir", type=Path, default=DATASET_DIR)
    parser.add_argument("--replace", action="store_true", help="Truncate tables before loading.")
    parser.add_argument("--only", nargs="*", help="Optional list of CSV stem/table names to load.")
    args = parser.parse_args([])

    mysql = require_connector()
    config = get_mysql_config()
    connection = mysql.connect(
        host=config.host,
        port=config.port,
        user=config.user,
        password=config.password,
        autocommit=False,
    )
    cursor = connection.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {q(config.database)}")
    cursor.execute(f"USE {q(config.database)}")

    dictionary = load_data_dictionary(args.dataset_dir / "data_dictionary.csv")
    csv_files = sorted(args.dataset_dir.glob("*.csv"))
    if args.only:
        wanted = set(args.only)
        csv_files = [path for path in csv_files if path.stem in wanted]

    for csv_path in csv_files:
        table = csv_path.stem
        loaded = load_csv(cursor, table, csv_path, dictionary.get(table, {}), args.replace)
        connection.commit()
        print(f"Loaded {loaded:>6} rows into {table}")

    apply_sql_file(cursor, Path("sql") / "analytics_views.sql", config.database)
    connection.commit()
    cursor.close()
    connection.close()
    print(f"Done. Database `{config.database}` is ready.")

In [ ]:
# ── Execute ──────────────────────────────────────────────────────────────────
main()

## 4. Train Oil Price Model
Builds feature/label pairs with a **1-day horizon**, trains a
`StandardScaler → Ridge` pipeline, and saves the model + metrics to
`model_artifacts/`.

Source: `src/train_oil_model.py`

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
from datetime import date
from pathlib import Path

import joblib
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# Raw columns read directly from the market CSV
FEATURE_COLUMNS = [
    "brent_price_usd",
    "wti_price_usd",
    "dxy_index",
    "vix_index",
    "gpr_index",
    "brent_return",
    "wti_return",
    "brent_lag_1",
    "brent_lag_3",
    "brent_lag_7",
    "wti_lag_1",
    "wti_lag_3",
    "wti_lag_7",
    "brent_volatility_7d",
    "brent_volatility_30d",
    "wti_volatility_7d",
    "wti_volatility_30d",
    "brent_wti_spread",
    "event_severity",
    "event_flag",
]

# Names for the computed features appended after the raw columns
COMPUTED_FEATURE_NAMES = [
    "brent_momentum_7d",   # absolute 7-day price change
    "brent_accel",         # recent vs week-ago momentum (direction signal)
    "vol_regime",          # short/long vol ratio (volatility regime indicator)
]


def to_float(value: str | None) -> float | None:
    if value is None or value == "":
        return None
    try:
        return float(value)
    except ValueError:
        return None


def compute_derived(vals: dict[str, float | None]) -> list[float | None]:
    p   = vals.get("brent_price_usd")
    l1  = vals.get("brent_lag_1")
    l7  = vals.get("brent_lag_7")
    v7  = vals.get("brent_volatility_7d")
    v30 = vals.get("brent_volatility_30d")

    momentum_7d = (p - l7)   if p  is not None and l7  is not None else None
    accel       = (l1 - l7)  if l1 is not None and l7  is not None else None
    vol_regime  = (v7 / v30) if v7 is not None and v30 is not None and v30 != 0.0 else None

    return [momentum_7d, accel, vol_regime]


def read_market_rows(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    return sorted(rows, key=lambda row: row["market_date"])


def build_examples(
    rows: list[dict[str, str]], horizon: int = 1
) -> tuple[list[list[float]], list[float], list[str]]:
    """Build feature/label pairs where the target is `horizon` trading days ahead."""
    x_rows: list[list[float]] = []
    y_rows: list[float] = []
    dates: list[str] = []
    for idx, row in enumerate(rows[: len(rows) - horizon]):
        future_brent = to_float(rows[idx + horizon].get("brent_price_usd"))
        raw_vals = {col: to_float(row.get(col)) for col in FEATURE_COLUMNS}
        raw_features = [raw_vals[col] for col in FEATURE_COLUMNS]
        computed = compute_derived(raw_vals)
        all_features = raw_features + computed
        if future_brent is None or any(v is None for v in all_features):
            continue
        x_rows.append([float(v) for v in all_features])
        y_rows.append(future_brent)
        dates.append(rows[idx + horizon]["market_date"])
    return x_rows, y_rows, dates


def split_chronological(
    x_rows, y_rows, dates, test_ratio
):
    split_index = max(1, int(len(x_rows) * (1.0 - test_ratio)))
    return (
        x_rows[:split_index], y_rows[:split_index], dates[:split_index],
        x_rows[split_index:], y_rows[split_index:], dates[split_index:],
    )


def metrics(actual: list[float], predicted: list[float]) -> dict[str, float]:
    return {
        "mae":      mean_absolute_error(actual, predicted),
        "rmse":     mean_squared_error(actual, predicted) ** 0.5,
        "mape_pct": mean_absolute_percentage_error(actual, predicted) * 100,
        "r2":       r2_score(actual, predicted),
    }


def write_predictions(
    path: Path,
    dates: list[str],
    actual: list[float],
    predicted: list[float],
    baseline: list[float],
    horizon: int = 1,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["market_date", "actual_brent_future", "predicted_brent_future",
                         "baseline_previous_brent", "horizon_days"])
        for d, a, p, b in zip(dates, actual, predicted, baseline):
            writer.writerow([d, a, p, b, horizon])


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Train one Ridge model per forecast horizon (direct multi-step strategy)."
    )
    parser.add_argument("--market-csv",   type=Path,  default=Path("datasets") / "ops_market_daily.csv")
    parser.add_argument("--output-dir",   type=Path,  default=Path("model_artifacts"))
    parser.add_argument("--test-ratio",   type=float, default=0.2)
    parser.add_argument("--alpha",        type=float, default=0.1,
                        help="Ridge regularization strength (default: 0.1).")
    parser.add_argument("--max-horizon",  type=int,   default=10,
                        help="Number of horizon models to train, 1 through N (default: 10).")
    args = parser.parse_args([])

    rows = read_market_rows(args.market_csv)
    args.output_dir.mkdir(parents=True, exist_ok=True)

    per_horizon: dict[int, dict] = {}
    # Capture h=1 artifacts for test_predictions.csv + backward-compat model
    h1_test_dates = h1_test_y = h1_test_preds = h1_baseline = None
    h1_train_metrics = h1_baseline_metrics = None

    print(f"Training {args.max_horizon} direct-horizon Ridge models "
          f"(alpha={args.alpha})  ...")

    for h in range(1, args.max_horizon + 1):
        x_rows, y_rows, dates = build_examples(rows, horizon=h)
        if len(x_rows) < 50:
            print(f"  h={h:02d}: not enough rows ({len(x_rows)}), skipping")
            continue

        train_x, train_y, train_dates, test_x, test_y, test_dates = split_chronological(
            x_rows, y_rows, dates, args.test_ratio
        )

        model = Pipeline([
            ("scaler",    StandardScaler()),
            ("regressor", Ridge(alpha=args.alpha)),
        ])
        model.fit(train_x, train_y)

        test_preds  = model.predict(test_x).tolist()
        train_preds = model.predict(train_x).tolist()
        m_test  = metrics(test_y,  test_preds)
        m_train = metrics(train_y, train_preds)

        model_file = args.output_dir / f"oil_price_model_h{h}.joblib"
        joblib.dump(model, model_file)

        per_horizon[h] = {
            "model_file":      str(model_file),
            "rmse":            round(m_test["rmse"],     4),
            "mae":             round(m_test["mae"],      4),
            "r2":              round(m_test["r2"],       6),
            "mape_pct":        round(m_test["mape_pct"], 4),
            "train_rows":      len(train_x),
            "test_rows":       len(test_x),
            "test_date_range": [test_dates[0], test_dates[-1]],
        }

        if h == 1:
            # Primary model (backward compat for anything that loads oil_price_model.joblib)
            joblib.dump(model, args.output_dir / "oil_price_model.joblib")
            h1_test_dates    = test_dates
            h1_test_y        = test_y
            h1_test_preds    = test_preds
            h1_baseline      = [row[0] for row in test_x]
            h1_train_metrics = m_train
            h1_baseline_metrics = metrics(test_y, h1_baseline)

        print(f"  h={h:02d}: RMSE={m_test['rmse']:.3f} USD  "
              f"MAE={m_test['mae']:.3f} USD  R²={m_test['r2']:.4f}")

    if not per_horizon:
        raise SystemExit("No horizon models trained — check dataset size.")

    # Write test_predictions.csv (h=1, for the visualization scatter / line chart)
    write_predictions(
        args.output_dir / "test_predictions.csv",
        h1_test_dates, h1_test_y, h1_test_preds, h1_baseline, horizon=1,
    )

    # Write model metadata JSON
    artifact = {
        "model_type":              "sklearn.pipeline.Pipeline(StandardScaler, Ridge)",
        "strategy":                "direct_multi_step",
        "max_horizon_trading_days": args.max_horizon,
        "horizon_trading_days":    1,            # kept for backward compat
        "trained_at":              date.today().isoformat(),
        "source_file":             str(args.market_csv),
        "feature_columns":         FEATURE_COLUMNS,
        "computed_feature_names":  COMPUTED_FEATURE_NAMES,
        "model_file":              str(args.output_dir / "oil_price_model.joblib"),
        "alpha":                   args.alpha,
        "per_horizon":             per_horizon,
        # h=1 summary fields kept for backward compat
        "train_rows":              per_horizon[1]["train_rows"],
        "test_rows":               per_horizon[1]["test_rows"],
        "train_date_range":        [h1_test_dates[0], h1_test_dates[-1]],
        "test_date_range":         per_horizon[1]["test_date_range"],
        "train_metrics":           h1_train_metrics,
        "test_metrics":            {k: v for k, v in per_horizon[1].items()
                                    if k in ("rmse", "mae", "r2", "mape_pct")},
        "baseline_previous_price_metrics": h1_baseline_metrics,
    }
    (args.output_dir / "oil_price_model.json").write_text(
        json.dumps(artifact, indent=2), encoding="utf-8"
    )

    print(f"\nSaved {len(per_horizon)} models to {args.output_dir}/")
    print(f"h=1  test RMSE : {per_horizon[1]['rmse']:.3f} USD   R2: {per_horizon[1]['r2']:.4f}")
    if args.max_horizon in per_horizon:
        print(f"h={args.max_horizon:02d} test RMSE : "
              f"{per_horizon[args.max_horizon]['rmse']:.3f} USD   "
              f"R2: {per_horizon[args.max_horizon]['r2']:.4f}")

In [ ]:
# ── Execute ──────────────────────────────────────────────────────────────────
main()

## 5. Forward Price Forecast
Uses a **Monte Carlo stochastic forecast**: 500 independent simulation paths
apply the h=1 Ridge model iteratively. At each step Gaussian noise
(sigma = h=1 test RMSE) is injected so paths diverge, producing a realistic
probability fan that widens naturally with horizon.

Output: `model_artifacts/forward_forecast.csv` — median + P10/P25/P75/P90.
Source: `src/predict_oil_price.py`

In [ ]:
"""predict_oil_price.py
Monte Carlo stochastic forecast of Brent crude oil prices.

Strategy: run N_SIMS independent simulation paths.  At each step the h=1
Ridge model predicts the next-day price; Gaussian noise (sigma = h=1 RMSE)
is added to represent prediction uncertainty.  Because each path diverges
independently the aggregate bands widen naturally with horizon.

Output columns in forward_forecast.csv:
    forecast_date, predicted_brent_usd (median), trading_days_ahead,
    p10, p25, p75, p90, source_date

Usage (from project root):
    python src/predict_oil_price.py
    python src/predict_oil_price.py --forecast-days 10 --n-sims 500
"""
from __future__ import annotations

import argparse
import csv
import json
import random as _random
from datetime import datetime, timedelta
from pathlib import Path

import joblib


# ---------------------------------------------------------------------------
# Helpers  (must stay in sync with train_oil_model.py)
# ---------------------------------------------------------------------------

def to_float(value: str | None) -> float:
    if value is None or value == "":
        raise ValueError("Missing numeric input.")
    return float(value)


def compute_derived(vals: dict[str, float]) -> list[float]:
    p   = vals.get("brent_price_usd")
    l1  = vals.get("brent_lag_1")
    l7  = vals.get("brent_lag_7")
    v7  = vals.get("brent_volatility_7d")
    v30 = vals.get("brent_volatility_30d")
    momentum_7d = (p  - l7) if p  is not None and l7  is not None else 0.0
    accel       = (l1 - l7) if l1 is not None and l7  is not None else 0.0
    vol_regime  = (v7 / v30) if v7 is not None and v30 is not None and v30 != 0.0 else 1.0
    return [momentum_7d, accel, vol_regime]


def estimate_future_trading_date(base_date_str: str, trading_days_ahead: int) -> str:
    dt = datetime.strptime(base_date_str, "%Y-%m-%d")
    added = 0
    while added < trading_days_ahead:
        dt += timedelta(days=1)
        if dt.weekday() < 5:
            added += 1
    return dt.strftime("%Y-%m-%d")


def _rolling_std(values: list[float]) -> float:
    n = len(values)
    if n < 2:
        return 0.0
    mean = sum(values) / n
    return (sum((x - mean) ** 2 for x in values) / (n - 1)) ** 0.5


def _buf_get(buf: list[float], n: int) -> float:
    idx = -(n + 1)
    return buf[idx] if len(buf) > n else buf[0]


def _percentile(sorted_vals: list[float], pct: float) -> float:
    idx = max(0, min(len(sorted_vals) - 1, int(pct * len(sorted_vals))))
    return sorted_vals[idx]


# ---------------------------------------------------------------------------
# Monte Carlo simulation
# ---------------------------------------------------------------------------

def monte_carlo_forecast(
    model,
    feature_columns: list[str],
    sorted_rows: list[dict[str, str]],
    forecast_days: int,
    n_sims: int,
    sigma: float,
    seed: int = 42,
) -> list[dict]:
    """
    Run `n_sims` independent stochastic forecast paths.

    Each path applies the h=1 Ridge model iteratively; at every step
    Gaussian noise N(0, sigma) is injected into the predicted price before
    it is fed back as the next input.  sigma = h=1 test-set RMSE, so the
    noise is calibrated to the model's actual 1-day prediction uncertainty.

    Returns one dict per forecast day with median + percentile statistics.
    """
    window   = 35
    recent   = sorted_rows[-window:]
    last_row = sorted_rows[-1]
    last_date = last_row["market_date"]

    brent_price_init  = [float(r["brent_price_usd"]) for r in recent if r.get("brent_price_usd")]
    wti_price_init    = [float(r["wti_price_usd"])   for r in recent if r.get("wti_price_usd")]
    brent_return_init = [float(r["brent_return"])     for r in recent if r.get("brent_return")]
    wti_return_init   = [float(r["wti_return"])       for r in recent if r.get("wti_return")]
    spread = float(last_row.get("brent_wti_spread", 0.0) or 0.0)

    anchor: dict[str, float] = {}
    for col in feature_columns:
        try:
            anchor[col] = to_float(last_row.get(col))
        except ValueError:
            anchor[col] = 0.0

    rng = _random.Random(seed)
    all_paths: list[list[float]] = []

    for _ in range(n_sims):
        brent_buf     = list(brent_price_init)
        wti_buf       = list(wti_price_init)
        brent_ret_buf = list(brent_return_init)
        wti_ret_buf   = list(wti_return_init)
        path: list[float] = []

        for step in range(1, forecast_days + 1):
            if step == 1:
                raw_vals    = {col: float(last_row.get(col, 0) or 0) for col in feature_columns}
                feature_vec = [raw_vals[col] for col in feature_columns] + compute_derived(raw_vals)
            else:
                brent     = brent_buf[-1]
                wti       = wti_buf[-1]
                brent_ret = brent_ret_buf[-1]
                wti_ret   = wti_ret_buf[-1]
                row: dict[str, float] = {}
                for col in feature_columns:
                    if   col == "brent_price_usd":     row[col] = brent
                    elif col == "wti_price_usd":       row[col] = wti
                    elif col == "brent_return":        row[col] = brent_ret
                    elif col == "wti_return":          row[col] = wti_ret
                    elif col == "brent_lag_1":         row[col] = _buf_get(brent_buf, 1)
                    elif col == "brent_lag_3":         row[col] = _buf_get(brent_buf, 3)
                    elif col == "brent_lag_7":         row[col] = _buf_get(brent_buf, 7)
                    elif col == "wti_lag_1":           row[col] = _buf_get(wti_buf, 1)
                    elif col == "wti_lag_3":           row[col] = _buf_get(wti_buf, 3)
                    elif col == "wti_lag_7":           row[col] = _buf_get(wti_buf, 7)
                    elif col == "brent_volatility_7d": row[col] = _rolling_std(brent_ret_buf[-7:])
                    elif col == "brent_volatility_30d":row[col] = _rolling_std(brent_ret_buf[-30:])
                    elif col == "wti_volatility_7d":   row[col] = _rolling_std(wti_ret_buf[-7:])
                    elif col == "wti_volatility_30d":  row[col] = _rolling_std(wti_ret_buf[-30:])
                    elif col == "brent_wti_spread":    row[col] = brent - wti
                    elif col in ("event_severity", "event_flag"): row[col] = 0.0
                    else:                              row[col] = anchor.get(col, 0.0)
                feature_vec = [row[col] for col in feature_columns] + compute_derived(row)

            pred_brent  = float(model.predict([feature_vec])[0])
            pred_brent += rng.gauss(0.0, sigma)   # stochastic noise
            path.append(pred_brent)

            prev_brent = brent_buf[-1]
            prev_wti   = wti_buf[-1]
            pred_wti   = pred_brent - spread
            brent_ret_new = (pred_brent - prev_brent) / prev_brent if prev_brent else 0.0
            wti_ret_new   = (pred_wti   - prev_wti)   / prev_wti   if prev_wti   else 0.0
            brent_buf.append(pred_brent)
            wti_buf.append(pred_wti)
            brent_ret_buf.append(brent_ret_new)
            wti_ret_buf.append(wti_ret_new)

        all_paths.append(path)

    # Aggregate across simulations
    forecasts: list[dict] = []
    for step_i in range(forecast_days):
        prices = sorted(p[step_i] for p in all_paths)
        forecasts.append({
            "forecast_date":       estimate_future_trading_date(last_date, step_i + 1),
            "predicted_brent_usd": round(_percentile(prices, 0.50), 4),
            "trading_days_ahead":  step_i + 1,
            "p10":                 round(_percentile(prices, 0.10), 4),
            "p25":                 round(_percentile(prices, 0.25), 4),
            "p75":                 round(_percentile(prices, 0.75), 4),
            "p90":                 round(_percentile(prices, 0.90), 4),
            "source_date":         last_date,
        })
    return forecasts


# ---------------------------------------------------------------------------
# Momentum blend  — makes the median less timid
# ---------------------------------------------------------------------------

def apply_momentum_blend(
    forecasts: list[dict],
    sorted_rows: list[dict[str, str]],
    momentum_window: int = 10,
    blend_weight: float = 0.4,
) -> list[dict]:
    """
    Correct for Ridge's mean-reversion bias by blending each step's MC median
    with a linear-trend extrapolation fitted to the last `momentum_window`
    actual trading days.

    blend_weight=0.0 -> pure MC median  (original behaviour)
    blend_weight=1.0 -> pure trend extrapolation
    blend_weight=0.4 -> 40 pct trend / 60 pct model (default)

    The entire distribution (p10/p25/p75/p90) is shifted by the same delta
    as the median, preserving the MC spread while moving the centre.
    """
    if blend_weight <= 0.0:
        return forecasts, 0.0

    recent = sorted_rows[-momentum_window:]
    prices = [float(r["brent_price_usd"]) for r in recent if r.get("brent_price_usd")]
    n = len(prices)
    if n < 2:
        return forecasts, 0.0  # not enough history

    # Ordinary least-squares slope over the window (price per trading day)
    xs = list(range(n))
    xm = sum(xs) / n
    ym = sum(prices) / n
    slope = sum((x - xm) * (y - ym) for x, y in zip(xs, prices)) / \
            sum((x - xm) ** 2 for x in xs)
    last_price = prices[-1]

    result: list[dict] = []
    for f in forecasts:
        step          = f["trading_days_ahead"]
        trend_price   = last_price + slope * step
        orig_median   = f["predicted_brent_usd"]
        blend_median  = (1.0 - blend_weight) * orig_median + blend_weight * trend_price
        shift         = blend_median - orig_median  # shift applied to all percentiles

        result.append({
            **f,
            "predicted_brent_usd": round(blend_median, 4),
            "p10": round(f["p10"] + shift, 4),
            "p25": round(f["p25"] + shift, 4),
            "p75": round(f["p75"] + shift, 4),
            "p90": round(f["p90"] + shift, 4),
        })
    return result, slope


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Monte Carlo stochastic forecast for Brent crude oil."
    )
    parser.add_argument("--metadata",        type=Path,  default=Path("model_artifacts") / "oil_price_model.json")
    parser.add_argument("--model",           type=Path,  default=None)
    parser.add_argument("--market-csv",      type=Path,  default=Path("datasets") / "ops_market_daily.csv")
    parser.add_argument("--forecast-days",   type=int,   default=10)
    parser.add_argument("--n-sims",          type=int,   default=500,
                        help="Number of Monte Carlo simulation paths (default: 500).")
    parser.add_argument("--seed",            type=int,   default=42)
    parser.add_argument("--momentum-blend",  type=float, default=0.4,
                        help="Weight given to the linear trend extrapolation "
                             "(0=pure MC, 1=pure trend, default: 0.4).")
    parser.add_argument("--momentum-window", type=int,   default=10,
                        help="Number of recent trading days used to estimate "
                             "the price trend (default: 10).")
    args = parser.parse_args([])

    artifact        = json.loads(args.metadata.read_text(encoding="utf-8"))
    model_path      = args.model or Path(artifact.get("model_file", "model_artifacts/oil_price_model.joblib"))
    feature_columns = artifact["feature_columns"]

    # Noise sigma: h=1 test-set RMSE (calibrated to actual 1-day prediction error)
    sigma = float(
        artifact.get("test_metrics", {}).get("rmse")
        or artifact.get("per_horizon", {}).get("1", {}).get("rmse")
        or 1.5
    )

    model = joblib.load(model_path)

    with args.market_csv.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    sorted_rows = sorted(rows, key=lambda r: r["market_date"])
    last_row    = sorted_rows[-1]

    print("=" * 60)
    print("  BRENT CRUDE OIL - MONTE CARLO FORECAST")
    print(f"  Paths: {args.n_sims}   Noise sigma: ${sigma:.2f}/day")
    print(f"  Momentum blend: {args.momentum_blend:.0%} trend  "
          f"({args.momentum_window}-day window)")
    print("=" * 60)
    print(f"  Last data date : {last_row['market_date']}")
    print(f"  Current Brent  : ${float(last_row['brent_price_usd']):.2f}")
    print(f"  Forecast window: {args.forecast_days} trading days")
    print("=" * 60)

    forecasts = monte_carlo_forecast(
        model, feature_columns, sorted_rows,
        args.forecast_days, args.n_sims, sigma, args.seed,
    )

    # Apply momentum blend to the aggregated medians
    forecasts, trend_slope = apply_momentum_blend(
        forecasts, sorted_rows,
        momentum_window=args.momentum_window,
        blend_weight=args.momentum_blend,
    )
    direction = "up" if trend_slope >= 0 else "down"
    print(f"  Trend slope    : ${trend_slope:+.3f}/day ({direction}ward)"
          f"  blend={args.momentum_blend:.0%}")

    print(f"\n{'Date':<14} {'Med ($)':<12} {'P10':<10} {'P25':<10} {'P75':<10} {'P90'}")
    print("-" * 66)
    for f in forecasts:
        print(f"{f['forecast_date']:<14} "
              f"${f['predicted_brent_usd']:<10.2f} "
              f"${f['p10']:<8.2f} ${f['p25']:<8.2f} "
              f"${f['p75']:<8.2f} ${f['p90']:.2f}")

    output_dir    = Path(model_path).parent
    forecast_path = output_dir / "forward_forecast.csv"
    output_dir.mkdir(parents=True, exist_ok=True)
    with forecast_path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=[
            "forecast_date", "predicted_brent_usd", "trading_days_ahead",
            "p10", "p25", "p75", "p90", "source_date",
        ])
        writer.writeheader()
        writer.writerows(forecasts)
    print(f"\nSaved {len(forecasts)}-day Monte Carlo forecast -> {forecast_path}")

In [ ]:
# ── Execute ──────────────────────────────────────────────────────────────────
main()

## 6. Visualizations
Generates **two interactive Plotly charts** (zoomable and pannable):

- **Prediction accuracy scatter** — actual vs predicted price correlation
- **Model vs Baseline vs Actual** — test-set time-series with the 10-day
  forward forecast bridged on as a dashed line with a shaded forecast window

Use the **1W / 2W / 1M … All** range buttons or drag the slider to zoom.

Source: `Visualization/visualize_predictions.py`

In [ ]:
"""visualize_predictions.py
Interactive Plotly visualizations for the Brent oil price model.

Produces two charts -- all fully zoomable and pannable with week-by-week
range-selector buttons:

  1. Prediction accuracy scatter (actual vs predicted)
  2. Model vs Baseline vs Actual (test set) + forward forecast extension
     with a shaded +/-1 sigma confidence band

The forward forecast line bridges from the last test-set date and extends
at least 5 trading days (one week) beyond the data cut-off, with a
confidence band derived from the per-horizon test-set RMSE.

Run standalone:
    python Visualization/visualize_predictions.py

Or import and call create_visualizations() from a notebook.
"""
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------

def _range_buttons() -> dict:
    """Week-by-week through All-time range selector buttons."""
    return dict(
        buttons=[
            dict(count=7,  label="1W",  step="day",   stepmode="backward"),
            dict(count=14, label="2W",  step="day",   stepmode="backward"),
            dict(count=1,  label="1M",  step="month", stepmode="backward"),
            dict(count=3,  label="3M",  step="month", stepmode="backward"),
            dict(count=6,  label="6M",  step="month", stepmode="backward"),
            dict(count=1,  label="1Y",  step="year",  stepmode="backward"),
            dict(step="all", label="All"),
        ],
        bgcolor="#f0f2f6",
        activecolor="#4a90d9",
    )


def _time_xaxis(title: str = "Date", x_range: list | None = None) -> dict:
    """Standard date x-axis with range selector and slider."""
    axis = dict(
        title=title,
        type="date",
        rangeselector=_range_buttons(),
        rangeslider=dict(visible=True, thickness=0.05),
        showgrid=True,
        gridcolor="#e5e5e5",
    )
    if x_range is not None:
        axis["range"] = x_range
    return axis


def _base_layout(**kwargs) -> dict:
    base = dict(
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=80, b=60),
        plot_bgcolor="#fafafa",
    )
    base.update(kwargs)
    return base


# ---------------------------------------------------------------------------
# Forecast loading
# ---------------------------------------------------------------------------

def _load_forecast(
    forecast_csv_path: str | None,
    last_test_date: pd.Timestamp,
    last_test_price: float,
) -> tuple:
    """
    Load forward_forecast.csv and build bridged series starting from the
    last actual test-set point (so forecast and test lines connect visually).

    Supports two CSV formats:
      - Monte Carlo format: columns p10 / p25 / p75 / p90
      - Legacy format: columns lower_1sigma / upper_1sigma

    Returns:
        bridge_dates, bridge_median,
        bridge_p10, bridge_p25, bridge_p75, bridge_p90,
        forecast_end
    All band series are None when not present in the CSV.
    """
    _empty = (None, None, None, None, None, None, None)
    if forecast_csv_path is None:
        return _empty
    try:
        fc = pd.read_csv(forecast_csv_path)
    except FileNotFoundError:
        return _empty
    if fc.empty:
        return _empty

    fc["forecast_date"] = pd.to_datetime(fc["forecast_date"])
    fc_sorted = fc.sort_values("forecast_date")

    def _bridge(series: pd.Series) -> pd.Series:
        """Prepend the anchor (last test-set) value so lines connect."""
        return pd.concat([pd.Series([last_test_price]), series.astype(float)],
                         ignore_index=True)

    bridge_dates  = pd.concat(
        [pd.Series([last_test_date]), fc_sorted["forecast_date"]],
        ignore_index=True,
    )
    bridge_median = _bridge(fc_sorted["predicted_brent_usd"])

    # Monte Carlo percentile bands
    if all(c in fc_sorted.columns for c in ("p10", "p25", "p75", "p90")):
        bridge_p10 = _bridge(fc_sorted["p10"])
        bridge_p25 = _bridge(fc_sorted["p25"])
        bridge_p75 = _bridge(fc_sorted["p75"])
        bridge_p90 = _bridge(fc_sorted["p90"])
    # Legacy +/-1sigma fallback
    elif "lower_1sigma" in fc_sorted.columns:
        bridge_p10 = bridge_p25 = _bridge(fc_sorted["lower_1sigma"])
        bridge_p75 = bridge_p90 = _bridge(fc_sorted["upper_1sigma"])
    else:
        bridge_p10 = bridge_p25 = bridge_p75 = bridge_p90 = None

    forecast_end = fc_sorted["forecast_date"].max()
    return bridge_dates, bridge_median, bridge_p10, bridge_p25, bridge_p75, bridge_p90, forecast_end


# ---------------------------------------------------------------------------
# Forecast traces
# ---------------------------------------------------------------------------

def _add_forecast_traces(
    fig,
    bridge_dates,
    bridge_median,
    bridge_p10,
    bridge_p25,
    bridge_p75,
    bridge_p90,
    last_test_date,
    forecast_end,
):
    """Draw the Monte Carlo probability fan: outer/inner bands + median line."""
    if bridge_p10 is not None:
        fig.add_trace(go.Scatter(
            x=bridge_dates, y=bridge_p90, mode="lines",
            line=dict(width=0), showlegend=False, hoverinfo="skip", name="_p90",
        ))
        fig.add_trace(go.Scatter(
            x=bridge_dates, y=bridge_p10, mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor="rgba(255,140,0,0.10)",
            showlegend=True, name="P10-P90 Range",
            hovertemplate="%{x|%Y-%m-%d}<br>P10: $%{y:.2f}<extra></extra>",
        ))
        fig.add_trace(go.Scatter(
            x=bridge_dates, y=bridge_p75, mode="lines",
            line=dict(width=0), showlegend=False, hoverinfo="skip", name="_p75",
        ))
        fig.add_trace(go.Scatter(
            x=bridge_dates, y=bridge_p25, mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor="rgba(255,140,0,0.22)",
            showlegend=True, name="P25-P75 Range",
            hovertemplate="%{x|%Y-%m-%d}<br>P25: $%{y:.2f}<extra></extra>",
        ))
    fig.add_trace(go.Scatter(
        x=bridge_dates, y=bridge_median,
        name="Forecast Median",
        line=dict(color="darkorange", width=2.5, dash="dash"),
        mode="lines+markers",
        marker=dict(size=6, color="darkorange", symbol="circle"),
        hovertemplate="%{x|%Y-%m-%d}<br>Median: $%{y:.2f}<extra></extra>",
    ))
    fig.add_vrect(
        x0=last_test_date, x1=forecast_end,
        fillcolor="darkorange", opacity=0.04, layer="below", line_width=0,
        annotation=dict(text="Forecast Window",
                        font=dict(size=11, color="darkorange"), align="left"),
        annotation_position="top left",
    )


# ---------------------------------------------------------------------------
# Chart builders
# ---------------------------------------------------------------------------

def _chart_scatter(
    df: pd.DataFrame,
    actual_col: str,
    predicted_col: str,
    horizon_label: str,
) -> go.Figure:
    min_val = min(df[actual_col].min(), df[predicted_col].min())
    max_val = max(df[actual_col].max(), df[predicted_col].max())

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df[actual_col],
        y=df[predicted_col],
        mode="markers",
        marker=dict(color="seagreen", opacity=0.45, size=6),
        name="Test Samples",
        hovertemplate="Actual: $%{x:.2f}<br>Predicted: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="crimson", dash="dash", width=1.5),
        name="Perfect Prediction (y = x)",
    ))
    fig.update_layout(**_base_layout(
        title=f"Prediction Accuracy: Actual vs Predicted ({horizon_label})",
        xaxis=dict(title="Actual Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
        yaxis=dict(title="Predicted Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
        hovermode="closest",
    ))
    return fig


def _chart_model_vs_baseline(
    df: pd.DataFrame,
    actual_col: str,
    predicted_col: str,
    horizon_label: str,
    bridge_dates: pd.Series | None,
    bridge_median: pd.Series | None,
    bridge_p10: pd.Series | None,
    bridge_p25: pd.Series | None,
    bridge_p75: pd.Series | None,
    bridge_p90: pd.Series | None,
    last_test_date: pd.Timestamp | None,
    forecast_end: pd.Timestamp | None,
) -> go.Figure:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df[actual_col],
        name="Actual Brent Price",
        line=dict(color="royalblue", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Actual: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df[predicted_col],
        name=f"Model Prediction ({horizon_label})",
        line=dict(color="darkorange", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Model: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df["baseline_previous_brent"],
        name="Baseline (Current Day Price)",
        line=dict(color="gray", width=1.5, dash="dash"),
        opacity=0.65,
        hovertemplate="%{x|%Y-%m-%d}<br>Baseline: $%{y:.2f}<extra></extra>",
    ))

    # Append forecast if available
    if bridge_dates is not None:
        _add_forecast_traces(
            fig, bridge_dates, bridge_median,
            bridge_p10, bridge_p25, bridge_p75, bridge_p90,
            last_test_date, forecast_end,
        )
        x_end = forecast_end + pd.Timedelta(days=3)
    else:
        x_end = df["market_date"].max() + pd.Timedelta(days=3)

    # Default zoom: show ~6 weeks of history + full forecast window
    # This guarantees the forecast (at least 1 week) is clearly visible
    six_weeks_before_end = x_end - pd.DateOffset(weeks=6)
    x_start = min(six_weeks_before_end, last_test_date - pd.DateOffset(weeks=4))

    fig.update_layout(**_base_layout(
        title=f"Model vs Baseline vs Actual -- {horizon_label} (Test Set + Forecast)",
        xaxis=_time_xaxis(x_range=[x_start, x_end]),
        yaxis=dict(title="Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
    ))
    return fig


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def create_visualizations(
    predictions_csv_path: str,
    output_dir: str | None = None,   # kept for API compatibility; no files written
    forecast_csv_path: str | None = None,
    market_csv_path: str | None = None,  # kept for API compatibility; unused
) -> None:
    """
    Generate two interactive Plotly charts for the Brent oil price model.

    Charts display inline in Jupyter or open in the browser when run
    standalone. No PNG files are written to disk.

    Parameters
    ----------
    predictions_csv_path : str
        Path to model_artifacts/test_predictions.csv (from train_oil_model.py).
    forecast_csv_path : str | None
        Path to model_artifacts/forward_forecast.csv (from predict_oil_price.py).
        If present the forecast line + confidence band are added to chart 2.
    output_dir, market_csv_path : ignored (backward compat).
    """
    print(f"Loading predictions from {predictions_csv_path} ...")
    try:
        df = pd.read_csv(predictions_csv_path)
    except FileNotFoundError:
        print(f"Error: {predictions_csv_path} not found. Train the model first.")
        return

    df["market_date"] = pd.to_datetime(df["market_date"])
    df = df.sort_values("market_date")

    # Horizon label
    horizon_days   = int(df["horizon_days"].iloc[0]) if "horizon_days" in df.columns else 1
    calendar_weeks = round(horizon_days / 5)
    week_str       = f"~{calendar_weeks} Week{'s' if calendar_weeks != 1 else ''}"
    horizon_label  = f"{horizon_days} Trading Day{'s' if horizon_days != 1 else ''} ({week_str}) Ahead"

    # Column name compatibility (old vs new schema)
    actual_col    = "actual_brent_future"    if "actual_brent_future"    in df.columns else "actual_brent_next"
    predicted_col = "predicted_brent_future" if "predicted_brent_future" in df.columns else "predicted_brent_next"

    # Bridge point: connect forecast from the last predicted price
    last_test_date            = df["market_date"].iloc[-1]
    last_test_predicted_price = float(df[predicted_col].iloc[-1])

    # Load forecast (Monte Carlo format with percentile bands)
    (
        bridge_dates, bridge_median,
        bridge_p10, bridge_p25, bridge_p75, bridge_p90,
        forecast_end,
    ) = _load_forecast(forecast_csv_path, last_test_date, last_test_predicted_price)
    if forecast_csv_path and bridge_dates is None:
        print("Warning: forecast CSV not loaded -- chart will show test set only.")

    # 1. Scatter: accuracy correlation
    fig1 = _chart_scatter(df, actual_col, predicted_col, horizon_label)
    fig1.show()

    # 2. Time-series: model vs baseline + Monte Carlo fan + median
    fig2 = _chart_model_vs_baseline(
        df, actual_col, predicted_col, horizon_label,
        bridge_dates, bridge_median,
        bridge_p10, bridge_p25, bridge_p75, bridge_p90,
        last_test_date, forecast_end,
    )
    fig2.show()


# ---------------------------------------------------------------------------
# Standalone entry point
# ---------------------------------------------------------------------------

In [ ]:
# ── Execute ───────────────────────────────────────────────────────────────────
# Paths mirror the defaults used when visualize_predictions.py is run standalone.
# Charts render inline as interactive Plotly HTML — no PNG files are written.
_predictions_csv = str(Path("model_artifacts") / "test_predictions.csv")
_forecast_csv    = str(Path("model_artifacts") / "forward_forecast.csv")

create_visualizations(
    _predictions_csv,
    forecast_csv_path=_forecast_csv,
)